# Notebook 05 — Petrobras 3W sector-agnostic contract challenge

This notebook asks the question that one telecom fixture cannot answer:

> Can a second sector be represented by a new pack and adapter without changing
> SPEC-CORE or adding telecom-specific branches?

It uses Petrobras 3W 2.0.0 from Drive. It has no public-download fallback.
The selection is the **smallest deterministic subset satisfying all criteria**, not
a fixed number of instances.

In [ ]:
from pathlib import Path
import json
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_DRIVE_ROOT = Path("/content/drive/MyDrive/anomaly_detection")
else:
    DEFAULT_DRIVE_ROOT = Path.cwd() / "anomaly_detection"

DRIVE_ROOT = Path(
    os.environ.get("ANOMALY_DETECTION_DRIVE_ROOT", str(DEFAULT_DRIVE_ROOT))
).expanduser()
CONTRACT_TAG = "v0.3"
CONTRACT_ROOT = DRIVE_ROOT / "contracts" / CONTRACT_TAG
PYTHON_SOURCE_ROOT = CONTRACT_ROOT / "python_src"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "milestone_1" / CONTRACT_TAG

print("Drive root:   ", DRIVE_ROOT)
print("Contract root:", CONTRACT_ROOT)
print("Output root:  ", OUTPUT_ROOT)

if not PYTHON_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Contract source not found at {PYTHON_SOURCE_ROOT}. Run Notebook 02 first."
    )
if str(PYTHON_SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE_ROOT))

In [ ]:
import hashlib
import resource
import tracemalloc
from datetime import datetime, timezone

import pandas as pd

from telemetry_adapters import ThreeWAdapter
from telemetry_contract import sha256_file

THREEW_SOURCE = Path(
    os.environ.get(
        "ANOMALY_DETECTION_3W_SOURCE",
        str(
            DRIVE_ROOT
            / "sources"
            / "petrobras_3w"
            / "2.0.0"
            / "raw"
            / "3w_dataset_2.0.0"
        ),
    )
).expanduser()
RUN_ID = os.environ.get("ANOMALY_DETECTION_3W_RUN_ID", "threew_contract_challenge_v1")
HASH_FULL_SOURCE = os.environ.get(
    "ANOMALY_DETECTION_HASH_FULL_3W_SOURCE", "1"
) not in {"0", "false", "False"}

RUN_ROOT = OUTPUT_ROOT / "petrobras_3w" / RUN_ID
CORE_OUTPUT = RUN_ROOT / "SPEC-CORE"
EVAL_OUTPUT = RUN_ROOT / "SPEC-EVAL"
FIXTURE_OUTPUT = (
    DRIVE_ROOT
    / "fixtures"
    / "petrobras_3w"
    / "2.0.0"
    / CONTRACT_TAG
    / RUN_ID
)

if not THREEW_SOURCE.is_dir():
    raise FileNotFoundError(
        f"3W source not found: {THREEW_SOURCE}\n"
        "Expected Drive path: anomaly_detection/sources/petrobras_3w/"
        "2.0.0/raw/3w_dataset_2.0.0"
    )
if RUN_ROOT.exists() or FIXTURE_OUTPUT.exists():
    raise FileExistsError("Choose a new RUN_ID; immutable output already exists.")

adapter = ThreeWAdapter()
inventory = adapter.discover(THREEW_SOURCE)
print(inventory.source_version, len(inventory.tables), inventory.notes)
assert inventory.core_ready and inventory.evaluation_ready
assert inventory.source_version == "2.0.0"
assert len(inventory.tables) == 2228

## 1. Pin the source

By default all 2,228 Parquet files are SHA-256 hashed. For a quick local smoke test
only, set `ANOMALY_DETECTION_HASH_FULL_3W_SOURCE=0`; the selected fixture files are
always hashed regardless.

In [ ]:
source_records = []
paths_to_hash = (
    [THREEW_SOURCE / relative for relative in inventory.tables]
    if HASH_FULL_SOURCE
    else []
)
for path in paths_to_hash:
    source_records.append(
        {
            "path": str(path.relative_to(THREEW_SOURCE)),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )
for metadata_name in ("dataset.ini", "LICENSE-CC-BY", "README.md"):
    path = THREEW_SOURCE / metadata_name
    source_records.append(
        {
            "path": metadata_name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )
print("Hashed source files:", len(source_records))

## 2. Select the smallest subset satisfying every challenge criterion

The subset must include normal data, a transient event, a persistent condition,
a state transition, a missing/frozen measurement case, and at least three distinct
real wells. The adapter searches deterministic Parquet-metadata candidates and
minimises the number of instances first, then total bytes.

In [ ]:
selected = adapter.select_minimal_subset(THREEW_SOURCE)
selection_table = pd.DataFrame(
    [
        {
            "path": item.relative_path,
            "entity_id": item.entity_id,
            "event_code": item.event_code,
            "rows": item.rows,
            "bytes": item.bytes,
            "coverage": ", ".join(sorted(item.coverage)),
        }
        for item in selected
    ]
)
display(selection_table)
assert len({item.entity_id for item in selected}) >= 3
assert all(item.source_kind == "real" for item in selected)

## 3. Translate and measure memory

No manifold topology, shared cause, ticket, or severity is invented. Native `class`
and `state` labels go only to SPEC-EVAL.

In [ ]:
def ru_maxrss_gib():
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return raw / (1024 ** 3) if sys.platform == "darwin" else raw * 1024 / (1024 ** 3)

rss_before = ru_maxrss_gib()
tracemalloc.start()
report = adapter.materialise(
    THREEW_SOURCE,
    CORE_OUTPUT,
    EVAL_OUTPUT,
    fixture_destination=FIXTURE_OUTPUT,
)
_, traced_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
rss_after = ru_maxrss_gib()

memory_report = {
    "tracemalloc_peak_gib": traced_peak_bytes / (1024 ** 3),
    "ru_maxrss_before_gib": rss_before,
    "ru_maxrss_after_gib": rss_after,
    "ru_maxrss_semantics": "process high-water mark; never falls in this kernel",
}
(RUN_ROOT / "memory_report.json").write_text(
    json.dumps(memory_report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(dict(report.row_counts), indent=2, sort_keys=True))
print(json.dumps(memory_report, indent=2))

## 4. Inspect the cross-sector finding

`gt_condition_states` is the one contract addition forced by the second sector.
`severity_ordinal` remains absent because 3W supplies class/state codes, not graded
component severity.

In [ ]:
condition_states = pd.read_parquet(
    EVAL_OUTPUT / "gt_condition_states.parquet"
)
relations = pd.read_parquet(CORE_OUTPUT / "entity_relations.parquet")
fit_report = json.loads(
    (RUN_ROOT / "threew_contract_fit_report.json").read_text(encoding="utf-8")
)
assert len(condition_states) > 0
assert "severity_ordinal" not in condition_states.columns
assert relations.empty
import pyarrow.parquet as pq
telemetry_columns = set()
for path in (CORE_OUTPUT / "telemetry").glob("part-*.parquet"):
    telemetry_columns.update(pq.ParquetFile(path).schema_arrow.names)
assert "class" not in telemetry_columns
assert "state" not in telemetry_columns
assert not any(str(column).startswith("gt_") for column in telemetry_columns)
display(condition_states)
print(json.dumps(fit_report, indent=2))

## 5. Write the versioned source and challenge manifest

In [ ]:
selected_manifest = json.loads(
    (FIXTURE_OUTPUT / "SUBSET_MANIFEST.json").read_text(encoding="utf-8")
)
challenge_manifest = {
    "contract_tag": CONTRACT_TAG,
    "source_version": inventory.source_version,
    "full_source_hashing_enabled": HASH_FULL_SOURCE,
    "source_files": source_records,
    "selected_fixture": selected_manifest,
    "contract_fit_report": fit_report,
    "truth_boundary": {
        "native_truth_fields": ["class", "state"],
        "destination": "SPEC-EVAL only",
    },
}
(RUN_ROOT / "source_and_challenge_manifest.json").write_text(
    json.dumps(challenge_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Notebook 05 complete:", RUN_ROOT)
print("Pinned fixture:", FIXTURE_OUTPUT)